In [21]:
from transformers import AutoTokenizer, AutoModel
import torch

# 选择一个嵌入模型，这里以 'sentence-transformers/all-MiniLM-L6-v2' 为例
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def embed_text(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        embeddings = model(**inputs, return_dict=True).pooler_output
    return embeddings.cpu().numpy()


In [8]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection

# 连接到 Milvus 服务
connections.connect("default", host="localhost", port="19530")

# 定义集合的 schema
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),  # dim需要和嵌入的维度匹配
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=512)
]
schema = CollectionSchema(fields, "text_embeddings")

# 创建集合
collection = Collection("text_collection", schema)


In [4]:
from pymilvus import utility
# 创建索引
index_params = {"index_type": "IVF_FLAT", "metric_type": "L2", "params": {"nlist": 128}}
collection.create_index("embedding", index_params)

Status(code=0, message=)

In [23]:
from datasets import load_dataset

# 加载数据集
dataset = load_dataset("allenai/ai2_arc",'ARC-Easy')

# 查看数据集的一部分
print(dataset['train'][0])


{'id': 'Mercury_7220990', 'question': 'Which factor will most likely cause a person to develop a fever?', 'choices': {'text': ['a leg muscle relaxing after exercise', 'a bacterial population in the bloodstream', 'several viral particles on the skin', 'carbohydrates being digested in the stomach'], 'label': ['A', 'B', 'C', 'D']}, 'answerKey': 'B'}


In [24]:
def assemble_question_answer(data):
    """
    根据输入字典组装问题和正确答案为一个字符串。

    参数:
    data (dict): 包含问题、选项和正确答案的字典。

    返回:
    str: 组装好的问题和正确答案字符串。
    """
    # 提取问题
    question = data['question']

    # 获取正确答案的标签
    answer_key = data['answerKey']

    # 根据标签找到正确的答案文本
    index = data['choices']['label'].index(answer_key)
    correct_answer = data['choices']['text'][index]

    # 组装成一个字符串
    result = f"{question} {correct_answer}"

    return result
print(assemble_question_answer(dataset['train'][0]))


Which factor will most likely cause a person to develop a fever? a bacterial population in the bloodstream


In [34]:
import json
import numpy as np
from tqdm import tqdm

def embed_data_to_json(dataset, output_file):
    data_list = []

    for i, item in tqdm(enumerate(dataset['train']), total=len(dataset['train']), desc="Processing embeddings"):
        text = assemble_question_answer(item)  # 组装问题和答案
        embedding = embed_text(text)  # 获取 GPU 上的嵌入张量

        # 转换为 NumPy 数组并确保数据类型为 float32
        embedding_np = embedding.cpu().numpy().astype(np.float32).tolist()[0] # 转换为 list 以便保存为 JSON

        # 构建保存的数据结构
        data_item = {
            'id': i,
            # 'question': item['question'],
            'embedding': embedding_np,
            'knowledge': text
        }

        data_list.append(data_item)

    # 保存为 JSON 文件
    with open(output_file, 'w') as f:
        json.dump(data_list, f, indent=4)

# 使用该函数将数据保存为 JSON
output_file = 'embeddings.json'
embed_data_to_json(dataset, output_file)


Processing embeddings: 100%|██████████| 2251/2251 [00:07<00:00, 315.53it/s]


In [35]:
from datasets import load_dataset
import json

# 加载数据集
dataset = load_dataset("allenai/ai2_arc", "ARC-Easy")

def get_incorrect_answers(data):
    question = data['question']
    correct_answer_key = data['answerKey']

    # 找到正确答案的索引
    correct_index = data['choices']['label'].index(correct_answer_key)

    # 获取所有非正确答案的文本
    incorrect_answers = [data['choices']['text'][i] for i in range(len(data['choices']['text'])) if i != correct_index]

    # 将问题和每个非正确答案组合成字符串
    result = [f"{question} {answer}" for answer in incorrect_answers]

    return result

# 处理整个数据集并将结果存储为列表
processed_data = []

for item in dataset['train']:
    incorrect_answers_str = get_incorrect_answers(item)
    processed_data.extend(incorrect_answers_str)  # 合并所有非正确答案的字符串

# 将结果保存为 JSON 文件
output_file = 'arc_easy_incorrect_answers.json'
with open(output_file, 'w') as f:
    json.dump(processed_data, f, indent=4)

print(f"Processed data saved to {output_file}")


Using the latest cached version of the dataset since allenai/ai2_arc couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'ARC-Easy' at C:\Users\Administrator\.cache\huggingface\datasets\allenai___ai2_arc\ARC-Easy\0.0.0\210d026faf9955653af8916fad021475a3f00453 (last modified on Thu Aug 29 11:21:35 2024).


Processed data saved to arc_easy_incorrect_answers.json
